<a href="https://colab.research.google.com/github/nvdpsingh/Qwen-3-8b-fine-tuned-for-finance-tool-calling/blob/main/project_tool_calling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

TOOL Calling Fine Tuning


In [1]:
!pip install unsloth trl datasets peft bitsandbytes accelerate anthropic -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 MB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 621.2/621.2 kB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 156.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.5/417.5 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 122.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 126.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get('HF_TOKEN'))

In [3]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Qwen3-8B-bnb-4bit",
    max_seq_length = 2048,
    dtype          = None,
    load_in_4bit   = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r                        = 16,
    lora_alpha               = 32,
    lora_dropout             = 0.05,
    target_modules           = ["q_proj", "v_proj", "k_proj", "o_proj",
                                 "gate_proj", "up_proj", "down_proj"],
    bias                     = "none",
    use_gradient_checkpointing = "unsloth",
    random_state             = 42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params : {trainable:,}")
print(f"Total params     : {total:,}")
print(f"Trainable %      : {100 * trainable / total:.3f}%")
print(f"GPU memory used  : {torch.cuda.memory_allocated() / 1e9:.2f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/6.07G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

unsloth/Qwen3-8B-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.4.4 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Trainable params : 43,646,976
Total params     : 4,761,498,624
Trainable %      : 0.917%
GPU memory used  : 6.29 GB


In [4]:
import json
from datasets import load_dataset, concatenate_datasets

# ── Load both datasets ────────────────────────────────────────────────
xlam_ds   = load_dataset("Salesforce/xlam-function-calling-60k", split="train")
domain_ds = load_dataset("json", data_files="financial_tool_calling_dataset.jsonl", split="train")

print(f"xLAM dataset   : {len(xlam_ds):,} examples")
print(f"Domain dataset : {len(domain_ds):,} examples")

# ── Format using Qwen3's correct chat template ────────────────────────
def format_example(row, is_domain=False):
    tools   = row["tools"]
    query   = row["query"]
    # xlam answers is a string, domain answers is already a list
    answers = row["answers"] if is_domain else row["answers"]
    if not isinstance(answers, str):
        answers = json.dumps(answers)
    if not isinstance(tools, str):
        tools = json.dumps(tools)

    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful AI assistant with access to the following tools:\n"
                f"{tools}\n\n"
                "When the user asks something that requires a tool, respond ONLY with "
                "a valid JSON function call. If no tool is needed, respond normally."
            )
        },
        {"role": "user",      "content": query},
        {"role": "assistant", "content": answers},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

xlam_formatted   = xlam_ds.map(lambda r: format_example(r, is_domain=False),
                                remove_columns=xlam_ds.column_names)
domain_formatted = domain_ds.map(lambda r: format_example(r, is_domain=True),
                                  remove_columns=domain_ds.column_names)

# ── Merge, shuffle, split 95/5 ────────────────────────────────────────
combined = concatenate_datasets([xlam_formatted, domain_formatted]).shuffle(seed=42)
split    = combined.train_test_split(test_size=0.05, seed=42)
train_ds = split["train"]
val_ds   = split["test"]

print(f"\nTrain : {len(train_ds):,}")
print(f"Val   : {len(val_ds):,}")

# ── Sanity check — must show <|im_start|>system structure ────────────
print("\nSample:\n")
print(train_ds[0]["text"])

README.md: 0.00B [00:00, ?B/s]

xlam_function_calling_60k.json:   0%|          | 0.00/96.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60000 [00:00<?, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

xLAM dataset   : 60,000 examples
Domain dataset : 2,100 examples


Map:   0%|          | 0/60000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2100 [00:00<?, ? examples/s]


Train : 58,995
Val   : 3,105

Sample:

<|im_start|>system
You are a helpful AI assistant with access to the following tools:
[{"name": "find_longest_word", "description": "Finds the longest word in a list of words.", "parameters": {"words": {"description": "A list of words.", "type": "List[str]"}}}, {"name": "flatten_list", "description": "Flattens a nested list into a single-level list.", "parameters": {"nested_list": {"description": "The nested list to be flattened.", "type": "List"}}}, {"name": "calculate_factorial", "description": "Calculates the factorial of a non-negative integer.", "parameters": {"n": {"description": "The non-negative integer.", "type": "int"}}}]

When the user asks something that requires a tool, respond ONLY with a valid JSON function call. If no tool is needed, respond normally.<|im_end|>
<|im_start|>user
Calculate the factorial of 7 and flatten the list [[1, [2, 3]], [4, 5], 6].<|im_end|>
<|im_start|>assistant
<think>

</think>

[{"name": "calculate_factori

In [5]:
from datasets import concatenate_datasets
from trl import SFTTrainer, SFTConfig

# ── Cap xLAM at 10k, keep all financial data ──────────────────────────
xlam_trimmed = xlam_formatted.shuffle(seed=42).select(range(10_000))
combined     = concatenate_datasets([xlam_trimmed, domain_formatted]).shuffle(seed=42)

split    = combined.train_test_split(test_size=0.05, seed=42)
train_ds = split["train"]
val_ds   = split["test"]

print(f"Train : {len(train_ds):,}")
print(f"Val   : {len(val_ds):,}")

# ── Trainer ───────────────────────────────────────────────────────────
trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = train_ds,
    eval_dataset  = val_ds,
    args          = SFTConfig(
        output_dir                  = "./qwen3-8b-toolcall",
        num_train_epochs            = 3,
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 2,
        gradient_checkpointing      = True,
        bf16                        = True,
        fp16                        = False,
        learning_rate               = 2e-4,
        warmup_ratio                = 0.05,
        lr_scheduler_type           = "cosine",
        weight_decay                = 0.01,
        logging_steps               = 25,
        eval_steps                  = 200,
        save_steps                  = 200,
        eval_strategy               = "steps",
        save_total_limit            = 2,
        load_best_model_at_end      = True,
        max_seq_length              = 2048,
        dataset_text_field          = "text",
        report_to                   = "none",
    ),
)

steps_per_epoch = len(train_ds) // (8 * 2)
total_steps     = steps_per_epoch * 3
print(f"Steps per epoch : {steps_per_epoch:,}")
print(f"Total steps     : {total_steps:,}")
print(f"Approx time     : ~{total_steps * 1.5 / 60:.0f} mins on A100")
print("\nStarting training...\n")

trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Train : 11,495
Val   : 605


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/11495 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/605 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 11,495 | Num Epochs = 3 | Total steps = 2,157
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 43,646,976 of 8,234,382,336 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
200,0.308998,0.305919
400,0.234777,0.245445
600,0.187819,0.195102
800,0.139845,0.159830
1000,0.110300,0.140213
1200,0.101592,0.121864
1400,0.097412,0.109144
1600,0.076660,0.106859
1800,0.073468,0.104311


Step,Training Loss,Validation Loss
200,0.308998,0.305919
400,0.234777,0.245445
600,0.187819,0.195102
800,0.139845,0.159830
1000,0.110300,0.140213
1200,0.101592,0.121864
1400,0.097412,0.109144
1600,0.076660,0.106859
1800,0.073468,0.104311
2000,0.070835,0.102903


TrainOutput(global_step=2157, training_loss=0.16520765407194846, metrics={'train_runtime': 7009.131, 'train_samples_per_second': 4.92, 'train_steps_per_second': 0.308, 'total_flos': 9.336374342683546e+17, 'train_loss': 0.16520765407194846, 'epoch': 3.0})

In [6]:
from unsloth import FastLanguageModel
import json

# ── Switch model to inference mode ────────────────────────────────────
FastLanguageModel.for_inference(model)

def run_inference(query, tools):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful AI assistant with access to the following tools:\n"
                f"{json.dumps(tools)}\n\n"
                "When the user asks something that requires a tool, respond ONLY with "
                "a valid JSON function call. If no tool is needed, respond normally."
            )
        },
        {"role": "user", "content": query},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize            = True,
        add_generation_prompt = True,
        return_tensors      = "pt",
    ).to("cuda")

    outputs = model.generate(
        input_ids        = inputs,
        max_new_tokens   = 256,
        temperature      = 0.1,       # low = more deterministic JSON
        do_sample        = True,
        pad_token_id     = tokenizer.eos_token_id,
    )

    # decode only the new tokens (not the prompt)
    response = tokenizer.decode(
        outputs[0][inputs.shape[1]:],
        skip_special_tokens = True
    )
    return response.strip()

# ── Test cases ────────────────────────────────────────────────────────
FINANCIAL_TOOLS = [
    {
        "name": "get_stock_price",
        "description": "Get real-time stock price for a ticker.",
        "parameters": {"ticker": {"type": "string"}, "currency": {"type": "string", "default": "USD"}}
    },
    {
        "name": "set_price_alert",
        "description": "Alert a user when a stock hits a target price.",
        "parameters": {"user_id": {"type": "string"}, "ticker": {"type": "string"},
                       "target_price": {"type": "number"}, "direction": {"type": "string"}}
    },
    {
        "name": "get_portfolio_summary",
        "description": "Get portfolio holdings and P&L for a user.",
        "parameters": {"user_id": {"type": "string"}, "include_unrealized": {"type": "boolean"}}
    },
    {
        "name": "search_news",
        "description": "Search recent financial news by keyword or ticker.",
        "parameters": {"query": {"type": "string"}, "max_results": {"type": "integer", "default": 5}}
    },
]

test_cases = [
    # Single tool — stock price
    "What's the current price of NVDA?",
    # Single tool — alert
    "Alert me when Tesla drops below $200. My user ID is user_1234.",
    # Multi tool
    "Get me the price of AAPL and search for recent news about Apple earnings.",
    # No tool needed
    "What is a P/E ratio?",
    # Edge case — ambiguous
    "How is my portfolio doing? I'm user_9999.",
]

print("=" * 60)
for i, query in enumerate(test_cases):
    response = run_inference(query, FINANCIAL_TOOLS)
    print(f"\nTest {i+1}: {query}")
    print(f"Response: {response}")

    # check if response is valid JSON
    try:
        parsed = json.loads(response)
        print(f"✅ Valid JSON — {len(parsed)} tool call(s)")
    except json.JSONDecodeError:
        print("💬 Not JSON (plain text response)")
    print("-" * 60)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Test 1: What's the current price of NVDA?
Response: <think>

</think>

[{"name": "get_stock_price", "arguments": {"ticker": "NVDA", "currency": "USD"}}]
💬 Not JSON (plain text response)
------------------------------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Test 2: Alert me when Tesla drops below $200. My user ID is user_1234.
Response: <think>

</think>

[{"name": "set_price_alert", "arguments": {"user_id": "user_1234", "ticker": "TSLA", "target_price": 200, "direction": "down"}}]
💬 Not JSON (plain text response)
------------------------------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Test 3: Get me the price of AAPL and search for recent news about Apple earnings.
Response: <think>

</think>

[{"name": "get_stock_price", "arguments": {"ticker": "AAPL"}}, {"name": "search_news", "arguments": {"query": "Apple earnings"}}]
💬 Not JSON (plain text response)
------------------------------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Test 4: What is a P/E ratio?
Response: <think>

</think>

[]
💬 Not JSON (plain text response)
------------------------------------------------------------

Test 5: How is my portfolio doing? I'm user_9999.
Response: <think>

</think>

[{"name": "get_portfolio_summary", "arguments": {"user_id": "user_9999", "include_unrealized": true}}]
💬 Not JSON (plain text response)
------------------------------------------------------------


In [7]:
import re, json

def parse_response(response):
    cleaned = re.sub(r"<think>.*?</think>", "", response, flags=re.DOTALL).strip()
    try:
        parsed = json.loads(cleaned)
        return parsed, True
    except json.JSONDecodeError:
        return cleaned, False

print("=" * 60)
for i, query in enumerate(test_cases):
    response = run_inference(query, FINANCIAL_TOOLS)
    parsed, is_json = parse_response(response)

    print(f"\nTest {i+1}: {query}")
    if is_json:
        print(f"✅ Valid JSON — {len(parsed)} tool call(s):")
        for call in parsed:
            print(f"   → {call['name']}({call.get('arguments', {})})")
    else:
        print(f"💬 Plain text: {parsed}")
    print("-" * 60)

Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Test 1: What's the current price of NVDA?
✅ Valid JSON — 1 tool call(s):
   → get_stock_price({'ticker': 'NVDA', 'currency': 'USD'})
------------------------------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Test 2: Alert me when Tesla drops below $200. My user ID is user_1234.
✅ Valid JSON — 1 tool call(s):
   → set_price_alert({'user_id': 'user_1234', 'ticker': 'TSLA', 'target_price': 200, 'direction': 'down'})
------------------------------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Test 3: Get me the price of AAPL and search for recent news about Apple earnings.
✅ Valid JSON — 2 tool call(s):
   → get_stock_price({'ticker': 'AAPL'})
   → search_news({'query': 'Apple earnings'})
------------------------------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Test 4: What is a P/E ratio?
✅ Valid JSON — 0 tool call(s):
------------------------------------------------------------

Test 5: How is my portfolio doing? I'm user_9999.
✅ Valid JSON — 1 tool call(s):
   → get_portfolio_summary({'user_id': 'user_9999', 'include_unrealized': True})
------------------------------------------------------------


In [9]:
HF_REPO = "nvdpsingh/qwen3-8b-financial-toolcall"

# ── Save LoRA adapter locally first ──────────────────────────────────
model.save_pretrained("qwen3-8b-toolcall-adapter")
tokenizer.save_pretrained("qwen3-8b-toolcall-adapter")
print("✅ Adapter saved locally")

# ── Push adapter to HF Hub (lightweight, ~150MB) ─────────────────────
model.push_to_hub(HF_REPO, token=userdata.get('HF_TOKEN'))
tokenizer.push_to_hub(HF_REPO, token=userdata.get('HF_TOKEN'))
print(f"✅ Adapter pushed to https://huggingface.co/{HF_REPO}")

# ── Merge LoRA into base weights → standalone model ───────────────────
model.save_pretrained_merged(
    "qwen3-8b-toolcall-merged",
    tokenizer,
    save_method = "merged_16bit",
)
print("✅ Adapters merged into full model")

# ── Push merged model to HF Hub ───────────────────────────────────────
model.push_to_hub_merged(
    HF_REPO + "-merged",
    tokenizer,
    save_method = "merged_16bit",
    token       = userdata.get('HF_TOKEN'),
)
print(f"✅ Merged model pushed to https://huggingface.co/{HF_REPO}-merged")

✅ Adapter saved locally


README.md:   0%|          | 0.00/541 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          |  556kB /  175MB            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0004-of-00004.safetensors:   7%|7         |  112MB / 1.58GB            


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [04:58<00:00, 74.71s/it]


Unsloth: Merge process complete. Saved to `/content/nvdpsingh/qwen3-8b-financial-toolcall-merged`
✅ Merged model pushed to https://huggingface.co/nvdpsingh/qwen3-8b-financial-toolcall-merged
